In [ ]:
import pandas as pd
import numpy as np
import pickle
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

from src.neuralnet import NeuralNetwork
from src.layers import DenseLayer, DropoutLayer
from src.activation import ReLUActivation, SoftmaxActivation

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    return text.strip()

df = pd.read_csv("data/dataset-exemplos.csv", sep=";")
df["clean_text"] = df["Text"].apply(clean_text)

# Label Encoding e One-Hot Encoding
le = LabelEncoder()
labels_idx = le.fit_transform(df["Label"])
y_onehot = np.zeros((len(labels_idx), len(le.classes_)))
y_onehot[np.arange(len(labels_idx)), labels_idx] = 1

# Extração de Features (TF-IDF)
vectorizer = TfidfVectorizer(max_features=2000, stop_words='english')
X = vectorizer.fit_transform(df["clean_text"]).toarray()

print(f"Formato dos dados (X): {X.shape}")
print(f"Formato das labels (y): {y_onehot.shape}")

In [ ]:
net = NeuralNetwork(epochs=30, batch_size=16, learning_rate=0.01)

net.add(DenseLayer(128, input_shape=(X.shape[1],)))
net.add(ReLUActivation())
net.add(DropoutLayer(drop_rate=0.3))
net.add(DenseLayer(64))
net.add(ReLUActivation())
net.add(DenseLayer(len(le.classes_)))
net.add(SoftmaxActivation())

print("A treinar o modelo NumPy...")
net.fit(X, y_onehot)

In [ ]:
with open("modelo_numpy_artefactos.pkl", "wb") as f:
    pickle.dump({
        "model": net, 
        "vectorizer": vectorizer, 
        "label_encoder": le
    }, f)
print("Modelo Numpy guardado com sucesso!")